## Data Cleaning Log

### Column selection (before cleaning)

| No. | Check | Reason | Action |
|---|---|---|---|
| A | Host-identifying columns | The project does not need to identify hosts, and these columns (names, IDs, profile URLs, descriptions, locations, pictures) are personal data | Excluded 10 host-identifying columns. |
| B | Other host-related columns | The analysis focuses on listing prices and property characteristics, not on hosts. `host_is_superhost` is an evaluative label about an individual host, so it was removed to stay consistent with the commitment not to profile or evaluate hosts | Excluded 18 other host-related columns, including `host_is_superhost`. |
| C | Remaining unused columns | Listing URLs and descriptions, availability, extra review metrics, price-quote fields and similar are not needed for the initial price comparison | Excluded the remaining unused columns, leaving the 14 columns used in the analysis. 76 of the 90 original columns were excluded in total. |

### Cleaning steps

| No. | Check | Reason | Action |
|---|---|---|---|
| 1 | Duplicate IDs | Duplicate listings would be counted twice and distort price comparisons | No duplicate IDs found. No action needed. |
| 2 | Price format | Prices were stored as text, so they could not be used in the analysis | Removed `$` signs and commas and converted the `price` column to `float64`. Checked 56,873 rows. |
| 3 | Missing prices | A listing without a valid price cannot be compared | Removed rows where `price` was missing after numeric conversion. All 5,009 were already missing in the raw file, so the conversion created no new missing values. 51,864 rows remained. |
| 4 | Bathroom format | `bathrooms_text` holds text such as "1 shared bath", which cannot be used in calculations | Created a numeric `bathrooms` column from `bathrooms_text`, counted after step 3 (51,864 rows). The first number in the text is used, so "3 baths" gives 3, "1 shared bath" gives 1, and "0 baths" gives 0. The 158 values without a number ("Half-bath", "Private half-bath", "Shared half-bath") were set to 0.5. The remaining 60 values could not be converted because `bathrooms_text` itself was missing, so they were kept as NaN. |
| 5 | Amenities | The amenities column holds a list of names as one text string, so it cannot be analyzed directly | Parsed each string as a list with `json.loads` and stored the number of items in `amenities_count`. All 51,864 values parsed without failure. 44 listings have an empty list and received a count of 0. Counts range from 0 to 100. |
| 6 | Missing bedrooms | Deleting these listings would lose about 15% of the rows, and the correct bedroom count cannot be inferred from the other columns | Counted after step 3: 7,859 missing values (8,513 before the missing-price rows were removed). The rows were kept and `bedrooms` was left as NaN. No values were filled in. |
| 7 | Invalid ratings | Ratings are on a 1 to 5 scale, so a value below 1 is invalid | Checked `review_scores_rating` for values below 1. Two listings (ids 1084850 and 1322855) had a rating of 0.0. The ratings were set to NaN and both listings were kept. |
| 8 | Missing ratings | A listing with no reviews has no rating, which is different from a rating of 0 | 14,501 ratings are NaN after this step: 14,499 listings with zero reviews (checked: none of them had a rating) plus the 2 values set to NaN in step 7. All of these listings were kept, and only their `review_scores_rating` is empty. |
| 9 | Missing minimum nights | One missing value is too few to justify removing the row, and no justified default exists | Found 1 missing `minimum_nights` value after step 3. The row was kept with NaN. It will need to be excluded or handled explicitly if `minimum_nights` is used later, not replaced with 0. |
| 10 | Long minimum stays | Stays longer than 365 nights are not typical short-term rentals and could distort price comparisons | 4 listings had `minimum_nights` above 365, but all had missing prices and were already removed in step 3. Added `minimum_nights_flag` (True above 365). No remaining rows are flagged. |
| 11 | Geographic coordinates | Listings outside Sicily and its islands would be outside the study area | Range check only: latitude between 35.4 and 38.9 and longitude between 11.8 and 15.7. These limits come from the approximate geography of Sicily and its outlying islands (for example Lampedusa and Pantelleria), not from an official boundary file. In the raw data (56,873 rows), latitude ranged from 35.49 to 38.81 and longitude from 11.93 to 15.65. A bounding box cannot confirm that a point is inside Sicily, since it also covers Malta and the southern tip of Calabria. No rows were outside the range, so none were removed. |
| 12 | High prices per guest | A few extreme prices would distort average prices. Price is divided by the number of guests so that large properties are not removed just for having a high total price | Calculated `price / accommodates` and removed listings above 2,000 USD per guest per night. The 7 removed listings ranged from about 2,792 to 61,161 USD per guest per night (listed below the table). The cutoff is a judgment based on inspecting the highest prices, not a confirmed error. 51,857 rows remained. |
| 13 | Low prices | A whole listing costing less than 10 USD per night is implausible whatever the number of guests, so the total price is checked. Step 12 uses price per guest instead because a high total price can be realistic for a large property | Applied to the total `price` per listing per night, not per guest. Removed 9 listings below 10 USD per night (listed below the table). The cutoff is a judgment based on inspecting the lowest prices, not a confirmed error. 51,848 rows remained. |
| 14 | Price outliers | Some very high prices are real (such as large properties) and the project is about finding overpriced listings, so extreme prices are flagged for further investigation instead of being deleted | Calculated Q1, Q3 and IQR of `price` separately for each `room_type`, using the 51,848 rows left after step 13. Flagged 4,724 listings above Q3 + 1.5 x IQR in `price_outlier` and kept them. The 1.5 multiplier is the conventional IQR rule for outliers, not a value tuned to this data. The thresholds are about 387.83 (Entire home/apt), 267.90 (Hotel room), 225.15 (Private room) and 81.25 (Shared room) USD per night. A flag marks an unusually high price within a room type; it does not prove that the listing is overpriced or that the price is wrong. Comparison across areas is done in the later analysis. |

**Final dataset:** 56,873 rows in, 51,848 rows out (5,025 removed). 14 columns in, 18 out (added bathrooms, amenities_count, minimum_nights_flag, price_outlier).

**Listings removed in step 12 (id, total price per night):** 20017976 (17,243.16), 43288399 (5,584.00), 52250690 (88,494.55), 823851890735056903 (61,160.55), 930598030535216768 (16,560.67), 1134674862349280891 (11,447.00), 1332491092213117150 (9,128.51).

**Listings removed in step 13 (id, total price per night):** 3611023 (2.59), 6605340 (2.02), 17859024 (9.96), 21453524 (8.08), 29648190 (8.37), 33969549 (8.11), 39637328 (8.31), 50144978 (7.35), 1137576175994870108 (9.53).